## Annotation automatique des images

Ce notebook permet de générer automatiquement des annotations pour chaque image du dossier `images/`. Il remplit deux objectifs principaux :

1. **Extraction des couleurs dominantes**  
   - Chaque image est convertie en vecteurs RGB.
   - Un algorithme de clustering (`KMeans`) est utilisé pour identifier les couleurs dominantes.
   - Les couleurs sont codées en hexadécimal (ex: `#aabbcc`).

2. **Extraction des métadonnées EXIF**  
   - Format de l’image, dimensions, mode colorimétrique.
   - Appareil photo utilisé, date de capture, ISO, exposition, focale.
   - Coordonnées GPS (si disponibles), converties en latitude / longitude.

Toutes ces informations sont combinées dans un fichier `annotations.json`, structuré par image. Ce fichier servira de base pour les étapes suivantes : analyse, visualisation, recommandation.

Le script ignore automatiquement les images :
- Trop volumineuses (> 50 Mo)
- Ne contenant pas de données exploitables
- Avec des erreurs de lecture ou d’EXIF

Ce traitement est entièrement automatisé.


In [ ]:
import os
import json
import cv2
import numpy as np
from PIL import Image, ExifTags
from sklearn.cluster import KMeans

IMAGE_FOLDER = "../images"
ANNOTATION_FILE = "JSON/annotations.json"

def convert_to_serializable(obj):
    try:
        return float(obj)
    except (TypeError, ValueError):
        return str(obj)

def safe_div(num, denom):
    try:
        return float(num) / float(denom) if denom != 0 else 0
    except:
        return 0

def get_exif_data(image):
    try:
        exif_data = image._getexif()
        if not exif_data:
            return {}
        return {
            ExifTags.TAGS.get(tag, tag): convert_to_serializable(value)
            for tag, value in exif_data.items()
        }
    except AttributeError:
        return {}

def get_gps_info(exif_data):
    if "GPSInfo" not in exif_data:
        return None
    gps_info = exif_data["GPSInfo"]
    if 2 not in gps_info or 4 not in gps_info:
        return None
    try:
        lat = [safe_div(x[0], x[1]) if isinstance(x, tuple) else float(x) for x in gps_info[2]]
        lon = [safe_div(x[0], x[1]) if isinstance(x, tuple) else float(x) for x in gps_info[4]]
        latitude = lat[0] + (lat[1] / 60.0) + (lat[2] / 3600.0)
        longitude = lon[0] + (lon[1] / 60.0) + (lon[2] / 3600.0)
        if gps_info.get(1) == 'S':
            latitude = -latitude
        if gps_info.get(3) == 'W':
            longitude = -longitude
        return {"latitude": latitude, "longitude": longitude}
    except Exception as e:
        print(f"Erreur GPS : {e}")
        return None

def get_dominant_colors(image_path, num_colors=3):
    img = cv2.imread(image_path)
    if img is None:
        return []
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (200, 200))
    pixels = img.reshape(-1, 3)
    kmeans = KMeans(n_clusters=num_colors, random_state=42).fit(pixels)
    return ["#{:02x}{:02x}{:02x}".format(int(r), int(g), int(b)) for r, g, b in kmeans.cluster_centers_]

def get_image_metadata(image_path):
    with Image.open(image_path) as img:
        exif_data = get_exif_data(img)
        gps_info = get_gps_info(exif_data)
        return {
            "format": img.format,
            "size": img.size,
            "mode": img.mode,
            "device": exif_data.get("Make", "Unknown") + " " + exif_data.get("Model", "Unknown"),
            "capture_date": exif_data.get("DateTimeOriginal", "Unknown"),
            "iso": exif_data.get("ISOSpeedRatings", "Unknown"),
            "focal_length": exif_data.get("FocalLength", "Unknown"),
            "exposure_time": exif_data.get("ExposureTime", "Unknown"),
            "gps": gps_info if gps_info else "No GPS Data"
        }

def process_images():
    annotations = {}
    for filename in os.listdir(IMAGE_FOLDER):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        full_path = os.path.join(IMAGE_FOLDER, filename)

        # Skip les très grosses images
        if os.path.getsize(full_path) > 50 * 1024 * 1024:
            print(f"Image trop lourde : {filename}, ignorée.")
            continue

        try:
            dominant_colors = get_dominant_colors(full_path)
            metadata = get_image_metadata(full_path)
            annotations[filename] = {
                "dominant_colors": dominant_colors,
                "metadata": metadata
            }
        except Exception as e:
            print(f"Erreur sur {filename} : {e}")
            continue

    with open(ANNOTATION_FILE, "w", encoding="utf-8") as f:
        json.dump(annotations, f, indent=4, ensure_ascii=False)

# Exécute tout
process_images()


Erreur sur 20130610%20-%20Monumento%20aos%20Her%C3%B3is%20da%20Independ%C3%AAncia.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Bottrop%20-%20Altmarkt%2002%20ies.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Cottbus%2007-2017%20img23%20Altmarkt.jpg : 'in <string>' requires string as left operand, not int


C:\Users\cmvil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\PIL\Image.py:3406: DecompressionBombWarning: Image size (175879204 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Erreur sur Fuerther%20Innenstadt%20Mrz%202020.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Koblenz%20-%20Panorama%20von%20Festung%20Ehrenbreitstein.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Lilongwe%20Area%202.jpg : division by zero
Erreur sur Moers%2C%20Schloss%2C%202024-07%20CN-01.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Nicosia%2001-2017%20img23%20View%20from%20Shacolas%20Tower.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Nyerere%20Square%20in%20Dodoma.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Panoramique%20du%20centre%20de%20Tunis%20%283197369985%29%20%28cropped%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Pointe%20des%20Almadies%20-%20Senegal.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Quartier%20OCH.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Recklingha